# BPI Challenge 2011: healthcare pathways

A patient can accumulate consultations, tests and treatment across several overlapping diagnosis-treatment phases. The practical question is where coordination becomes difficult while indicated care and patient access remain protected.

The questions below concern the complete recorded case. WISE evaluates the declared constraints and returns the numerical summaries, priorities and evidence. Figures display those returned results. The review scales are exploratory inputs awaiting process-owner approval; a flagged case is a starting point for investigation.

## PM² route

The work follows the six stages of [PM²](https://vdaalst.com/publications/p824.pdf). Revisit earlier stages when evidence changes the question or its scope.

| Stage | Work recorded here | Decision required |
|---|---|---|
| 1. Planning | Business goal, case unit, interpretation questions and candidate norms | Confirm the investigation scope and acceptable service outcomes |
| 2. Extraction | Read the supplied challenge logs and identify source versions | Check the extraction boundary and missing records |
| 3. Data processing | Retain whole cases, review activity bindings and context | Resolve ambiguous meanings and episode boundaries |
| 4. Mining and analysis | Review severity, coverage, context, priorities and original traces | Identify a recurring mechanism supported by evidence |
| 5. Evaluation | Compare views, settings and plausible explanations | Decide whether a proposed change warrants a trial |
| 6. Process improvement and support | Assign an owner, define a pilot and protect essential work | Measure the result before expanding the change |

The execution journal records the analytical run. The review journal near the end records business judgements separately; running a notebook does not approve a norm or complete a process-improvement trial.

In [ ]:
import os
from pathlib import Path
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import wise

try:
    from bpic_analysis import (
        challenge, summary, norm_table, criterion_summary, activity_summary,
        context_profiles, priorities, plot_overview, case_evidence,
        threshold_sensitivity, view_agreement, integrity, save_outputs, journal,
    )
except ModuleNotFoundError as error:
    if error.name != "bpic_analysis":
        raise
    from notebooks.bpic_analysis import (
        challenge, summary, norm_table, criterion_summary, activity_summary,
        context_profiles, priorities, plot_overview, case_evidence,
        threshold_sensitivity, view_agreement, integrity, save_outputs, journal,
    )

if wise.__version__ != "1.0.0":
    raise RuntimeError("Select the environment with the frozen 1.0.0 package.")

YEAR = 2011
data_root = os.environ.get("WISE_PAPER_DATA")
SAVE_OUTPUTS = False


def show_evidence(evidence):
    for label, value in evidence.items():
        display(Markdown("**" + label.replace("_", " ") + "**"))
        display(value)


def show_figures(run):
    for figure in plot_overview(run):
        display(figure)
        plt.close(figure)

## 1. Planning: the business question

**Goal:** reduce avoidable coordination waiting and repeated requests for the same information while retaining required care.

**Case unit:** one recorded patient trace. It can contain several diagnosis-treatment combinations, so one trace does not automatically describe one clinical episode.

Investigate three questions:

1. Which whole-case diagnosis groups contain concentrated contact or follow-up burden under the stated review scales?
2. Does that burden reflect planned surveillance, multiple treatment phases, patient support, or avoidable routing and scheduling uncertainty?
3. Where is a timing comparison unsupported because episode identity or timestamp precision is insufficient?

A suitable outcome would be verified reduction in avoidable coordination delay with unchanged clinical follow-up and patient access. Consultation counts alone cannot establish this outcome.

In [ ]:
runs = challenge(YEAR, data_folder=data_root)
display(summary(runs))

for identifier, run in runs.items():
    if run["record"]["status"] != "computed":
        display(run["record"])
        raise RuntimeError("Resolve the recorded source or activity-binding issue before analysis.")
    display(Markdown("### " + identifier))
    display(run["specification"]["business_goal"])

## 2. Extraction: establish the evidence boundary

The supplied log comes from a Dutch academic hospital. Activity names are Dutch and records include organisational groups. The raw metadata describe patient traces and completed service-event records; an event lifecycle of `complete` does not establish completion of the patient pathway.

Use source event rows as the counting unit. A `Number of executions` field can express another source quantity and must not be silently substituted for row count. Keep the source values available in the case drill-down. Date-only service timestamps cannot justify minute-level waiting estimates. Multiple records on the same date do not establish clinical ordering.

Source fingerprints and dataset counts appear in the execution summary. The integrity record below checks numerical norm round-trip agreement, signed decomposition, case-index agreement and the count of events retained by the log object. It is not an independent certification of raw-source completeness or event-by-event retention. Criterion-level availability appears later. These numerical checks do not establish clinical and operational correctness.

In [ ]:
integrity_records = {}
for identifier, run in runs.items():
    integrity_records[identifier] = integrity(run)
    display(Markdown("### " + identifier))
    display(integrity_records[identifier])

## 3. Data processing: preserve whole-case context

Inspect `Diagnosis`, `Age` and the numbered diagnosis-treatment attributes in the source case record. A base `Diagnosis` field is an exploratory grouping, not a complete account of every phase. The trace can also contain `Diagnosis:1`, `Diagnosis:2` and matching combination identifiers and dates. Any inference about an episode must use its own validated linkage.

The event and trace can both have a `Specialism code`; preserve their separate meanings. `org:group` describes an organisational group rather than an identified clinician. Differences between diagnosis groups may reflect case mix and treatment intention.

Read the profile of missing and mixed context before comparing groups. The case-start and last-observed month labels use the earliest and latest usable recorded timestamps after UTC normalisation. They can differ from business-local calendar months and do not verify the actual process start or finish. Verified completion needs a reviewed terminal-state definition. The observed span is the interval between these recorded endpoints; it does not establish processing time. Calendar selections must retain all events of each selected case.

In [ ]:
context_tables = {}
for identifier, run in runs.items():
    context_tables[identifier] = context_profiles(run, attributes=['Diagnosis', 'Age', 'Diagnosis code', 'Specialism code', 'Diagnosis Treatment Combination ID'])
    display(Markdown("### " + identifier))
    display(context_tables[identifier])

### Turn goals into inspectable review criteria

The initial review criteria examine repeated first outpatient consultations, telephone-contact volume, follow-up volume and the first recorded outpatient-to-follow-up interval. These are coordination and pathway-complexity screens.

A positive follow-up interval can describe planned scheduling. Several initial consultations can correspond to different legitimate phases. The timing range in the table has no standing as a clinical deadline. An episode-specific clinical follow-up norm remains pending until treatment intention, scheduling protocol and patient-specific exceptions are known.

The table exposes the exact labels, scales, applicability, missing-data policy and stakeholder views used in the run. For a lag, the baseline uses the earliest usable start and the first usable end at or after it. It does not pair every cycle. Missing endpoints are unavailable evidence. Occurrence criteria count recorded event rows; they do not prove that repeated work was unnecessary. The Balanced view assigns equal total weight to each declared layer.

In [ ]:
for identifier, run in runs.items():
    display(Markdown("### " + identifier))
    display(norm_table(run))
    display(Markdown("**Questions awaiting additional semantics or policy evidence**"))
    display(run["specification"].get("pending_checks", []))

## 4. Mining and analysis: distinguish activity volume from case burden

Compare initial outpatient consultations, telephone contacts and follow-up consultations. A large occurrence count with a much smaller case count suggests repeated recording within a subset of pathways. Return to those pathways before deciding whether the repetitions are necessary care or coordination work.

The first measure counts recorded occurrences. The second counts cases with at least one occurrence. Their units differ. Neither is an estimate of staff effort, avoidable cost or a causal explanation.

In [ ]:
activity_tables = {}
for identifier, run in runs.items():
    table = activity_summary(run, labels=['1e consult poliklinisch', 'telefonisch consult', 'vervolgconsult poliklinisch'])
    activity_tables[identifier] = table
    display(Markdown("### " + identifier))
    display(table)
    if not table.empty:
        figure, axes = plt.subplots(1, 2, figsize=(14, 5), squeeze=False)
        table.set_index("activity")[["occurrences"]].plot.barh(
            ax=axes[0, 0], legend=False, color="C0"
        )
        table.set_index("activity")[["observed_cases"]].plot.barh(
            ax=axes[0, 1], legend=False, color="C1"
        )
        axes[0, 0].set_title("Recorded occurrences")
        axes[0, 1].set_title("Cases with an occurrence")
        axes[0, 0].set_xlabel("Event rows")
        axes[0, 1].set_xlabel("Whole cases")
        for axis in axes[0]:
            axis.xaxis.set_major_locator(MaxNLocator(5))
        figure.suptitle(identifier)
        figure.tight_layout()
        display(figure)
        plt.close(figure)

### Severity and evidence availability answer different questions

Compare the timing criterion with the three occurrence criteria. Incomplete endpoint evidence can make the timing result unavailable even when consultation volume is observed. A patient with many appropriate follow-ups can legitimately trigger the complexity screen; this calls for explanation, not a proposal to remove care.

Inspect evaluated and in-scope counts beside mean severity. Evidence coverage is the fraction of in-scope cases that support a criterion evaluation. A low severity with little available evidence is inconclusive. A high severity under an illustrative scale still needs its process meaning checked.

**Measured baseline in the supplied log.** The first outpatient-to-follow-up interval is evaluable in 523 of 1,143 cases. The library reports 440 cases with missing activation evidence and 180 with missing response evidence under this endpoint binding. These are evidence states, not counts of patients who failed to receive required care.

The base `Diagnosis` field is missing in 455 cases, and this group leads the Balanced relative-priority list. Source witness `00000294` nevertheless records `Diagnosis:1`, `Diagnosis:2` and `Diagnosis:3` with phase identifiers. A missing base field therefore does not establish absence of a recorded clinical diagnosis. Review phase-aware context and endpoint bindings before attributing the group to poor coordination or delayed care.

These findings describe the measured source and norm snapshot. Their fingerprints are printed beside the table below; rerun the analysis after either changes.

Per-criterion coverage in this table is an unweighted fraction of in-scope cases. The source witness also shows each view's weighted case-level evidence coverage, which measures the available share of its in-scope criterion weight. Scores renormalise the usable evidence, so a high returned score with incomplete coverage leaves parts of the declared norm unassessed. Neither coverage measure certifies source completeness.

In [ ]:
criterion_tables = {}
for identifier, run in runs.items():
    table = criterion_summary(run)
    criterion_tables[identifier] = table
    display(Markdown("### " + identifier))
    display({"source_sha256": run["record"]["source_sha256"],
             "norm_fingerprint": run["record"]["norm_fingerprint"]})
    display(table)
    if not table.empty:
        axis = table.set_index("criterion_id")[["evidence_coverage", "mean_severity"]].plot.bar(
            figsize=(10, 4), ylim=(0, 1), rot=0
        )
        axis.set_ylabel("Fraction or mean severity")
        axis.set_title(identifier + ": evidence and review severity")
        axis.figure.tight_layout()
        display(axis.figure)
        plt.close(axis.figure)

### Choose a context for the investigation list

Start with the declared whole-case diagnosis grouping. Compare the Timing and Complexity views to see whether the investigation depends on a preference for interval length or recorded contact burden. Review `Age` only as a descriptive case attribute; this run does not adjust for disease severity or clinical risk.

Read the returned grouping and view specification beside each list. Priority combines case volume and a score gap relative to the population of this run. A zero relative priority does not establish an acceptable absolute result. Small-slice shrinkage is a declared sensitivity setting, not a fitted service target.

In [ ]:
investigation_lists = {}
for identifier, run in runs.items():
    investigation_lists[identifier] = {}
    for view in run["result"].views:
        backlog = priorities(run, view=view, gamma=0)
        adjusted = priorities(run, view=view, gamma=20)
        investigation_lists[identifier][view] = backlog
        display(Markdown("### " + identifier + " / " + view))
        display(backlog.spec.to_dict())
        display(backlog.table.head(10))
        display(Markdown("**Same population with small-slice shrinkage**"))
        display(adjusted.table.head(10))
    for grouping in ['Diagnosis', 'Age']:
        backlog = priorities(run, by=grouping, gamma=0)
        display(Markdown("**Alternative whole-case grouping: " + str(grouping) + "**"))
        display(backlog.spec.to_dict())
        display(backlog.table.head(10))

### Read the overview together with its signed explanation

Look for a group whose apparent coordination burden persists across views, then inspect whether a negative timing contribution offsets its repeated-contact contribution. Different signs provide a reason to question a simple claim that all aspects of that pathway are worse.

These figures use the evaluated cases and library-returned priorities and contributions. Positive layer contributions add to the relative penalty gap; negative contributions offset it. Preserve both signs when explaining a shortlisted group. The decomposition describes the score, not the cause of a process problem.

In [ ]:
for identifier, run in runs.items():
    display(Markdown("### " + identifier))
    show_figures(run)

### Return to the actual case

Inspect separate witnesses for telephone contacts and the first follow-up interval. Locate all diagnosis-treatment combinations in each source case. Ask whether contacts belong to the same phase, whether their purpose is recorded and whether follow-up timing was planned. An event sequence alone cannot resolve those questions.

The helper chooses a recorded witness for the requested criterion and returns the original source events, case context and evaluated criterion evidence. The full source trace remains available. Read rows with tied timestamps as unresolved ordering unless the source provides a stronger order. To inspect another case, pass its actual identifier as `case_id` to `case_evidence`.

In [ ]:
run = runs["bpic2011"]
for criterion in ("H11_02", "H11_04"):
    display(Markdown("### Source witness: " + criterion))
    show_evidence(case_evidence(run, criterion=criterion))

### Place the case evidence on a timeline

This figure uses event timestamps returned by the library for a complete recorded case selected by the first diagnostic criterion. Named lanes highlight norm activities; the grey lane keeps the remaining recorded activities visible. Coincident points can overlap and establish no order. Undated events remain in the source table and cannot be placed on the time axis. The selected case is an investigation example, not a representative sample; return to its full source witness before interpreting an apparent gap.

In [ ]:
from bpic_analysis import plot_case_timeline
for identifier, run in runs.items():
    if "result" not in run:
        continue
    figure = plot_case_timeline(run)
    display(figure)
    plt.close(figure)

## 5. Evaluation: challenge the interpretation

A shortlist that changes substantially with the review scale is sensitive to an unapproved input. A shortlist that persists can justify closer review, but its clinical interpretation still needs episode linkage.

Before labelling waiting as avoidable, distinguish planned surveillance intervals from delays in arranging required work. Before labelling contacts as duplication, distinguish patient support and phase transitions from repeat routing for the same unresolved request.

The factors below scale both the lag threshold `delta` and saturation width `width`. For occurrence criteria they scale the saturation width `K` while keeping the integer occurrence tolerance `k` fixed. The helper rescores each declared scenario with WISE. These tests examine the chosen scales; they do not estimate an optimal policy. The view agreement table compares positive-priority shortlist membership under the declared views; it does not measure causal confidence or validate a business policy. Inspect a large disagreement by returning to its evidence.

In [ ]:
sensitivity_tables = {}
view_tables = {}
for identifier, run in runs.items():
    display(Markdown("### " + identifier))
    sensitivity_tables[identifier] = threshold_sensitivity(run, factors=(0.5, 1, 2))
    view_tables[identifier] = view_agreement(run, k=10)
    display(sensitivity_tables[identifier])
    display(view_tables[identifier])

## 6. Process improvement and support: define a feasible trial

| Confirmed mechanism required | Feasible change and owner | Pilot measure | Protected outcome |
|---|---|---|---|
| The same coordination request repeatedly reaches different teams without a clear owner | Pathway coordinator trials one named contact, shared request status and an agreed handoff rule | Unanswered requests and verified time to an agreed next step | Access to all required specialists and patient support |
| Several teams repeatedly ask for the same already available information | Service manager trials one reviewed information checklist and reuse of the source record | Repeat requests for identical information | Clinical completeness and correction of inaccurate records |
| A required follow-up is delayed by scheduling uncertainty | Scheduling lead trials an explicit due review and escalation owner under the approved clinical plan | Verified booking delay within comparable treatment phases | Indicated follow-up, patient choice and access |

Do not trial fewer consultations or tests from an occurrence score alone. First review a small set of actual cases with the pathway team and classify the reason for repeated work. Expand only a confirmed coordination change.

A change proceeds after the mechanism has been confirmed in source-record review and its owner has accepted the scope. Declare the baseline cohort, observation horizon, exceptions, outcome measures and stop conditions before the pilot. Record changes in case mix and evidence collection. A shorter recorded trace or lower diagnostic score alone does not establish a better process.

### Full analysis journal and review record

The execution journal below covers the run and its evidence checks. Add dated observations to `review_journal` as cases are discussed. Each entry should identify its supporting case or table, a plausible competing explanation, the reviewer decision and the next action. Empty fields remain unfilled work; they are not invented findings.

In [ ]:
execution_journal = journal(runs)
display(execution_journal)

review_journal = {
    "business_goal_and_population": "",
    "norms_approved_or_changed": "",
    "evidence_identifiers_reviewed": [],
    "observed_pattern": "",
    "competing_explanations": [],
    "confirmed_avoidable_mechanism": "",
    "decision_and_owner": "",
    "pilot_scope_and_comparison": "",
    "protected_outcomes_and_stop_conditions": [],
    "review_date": "",
    "next_action": "",
}
display(review_journal)

### Decision gates before reporting an improvement

- Has the patient trace been separated into reviewed treatment phases where the question requires it?
- Are planned care and avoidable coordination distinguishable in the inspected records?
- Are clinical scheduling rules and patient-specific exceptions approved by the responsible team?
- Can the pilot measure a coordination outcome without treating less recorded care as improvement?

The portable output option saves the actual run records, tables and figures in `outputs`. Review case-level information before sharing saved outputs. Cell outputs, when present, describe the last saved execution. Rerun all cells after input or norm changes.

In [ ]:
if SAVE_OUTPUTS:
    output_folder = Path("outputs") / "bpic2011"
    save_outputs(runs, output_folder)
    display({"saved_outputs": output_folder.as_posix()})

## Source and method notes

- [BPI Challenge 2011 process description](https://ais.win.tue.nl/bpi/2011/challenge.html) and the supplied source metadata.
- [Public hospital event log](https://doi.org/10.4121/uuid:d9769f3d-0ab0-4fb8-803b-0d1120ffcf54).
- [PM² methodology](https://vdaalst.com/publications/p824.pdf).

The declared norm and source fingerprints are recorded by the helper. WISE calculates constraint evaluations, scores, priorities and decompositions; notebook code selects and presents those results. Pending rules require the additional evidence stated in the norm specification.